# 005 Triple Extraction With Qwen

这是 RAG 知识库学习线的第五课。

上一课我们把 chunk 写入 ES 的流程设计好了。本课进入 GraphRAG 的知识转化阶段：

```text
chunks.json
-> qwen2.5-0.5b-instruct
-> entities + triples
-> triples.json
```

学习目标：

1. 理解三元组抽取在 GraphRAG 中的位置。
2. 使用 `qwen2.5-0.5b-instruct` 抽取实体关系。
3. 学会设计固定 JSON Schema 的抽取 prompt。
4. 学会解析、修复和校验小模型输出。
5. 为 Neo4j 入库准备 `triples.json`。

注意：小模型可能漏字段或输出不稳定。本课的重点不是“相信模型”，而是“模型输出后必须校验”。

## 1. 三元组是什么

三元组是图谱里最基础的关系表达：

```text
subject --predicate--> object
```

例如：

```text
密云水库管理处 --发布--> 泄洪通知
泄洪通知 --要求--> 下游乡镇做好防汛准备
```

进入 Neo4j 前，我们还要保留证据：

```text
evidence
confidence
doc_id
chunk_id
page_start
page_end
```

否则图谱就会变成“没有出处的结论库”。

## 2. 导入依赖

本课主要使用：

```text
openai -> 调 qwen2.5-0.5b-instruct
json   -> 解析模型输出
```

In [27]:
import importlib.metadata
import json
import os
import re
from hashlib import sha1
from pathlib import Path
from pprint import pprint

from dotenv import load_dotenv
from openai import OpenAI

print('openai', importlib.metadata.version('openai'))

openai 2.36.0


## 3. 加载 chunks.json

第五课从第三课产出的 `chunks.json` 开始。

如果找不到这个文件，请先执行第三课。

In [28]:
def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for path in [current, *current.parents]:
        if (path / 'requirements.txt').exists() and (path / 'notebooks').exists():
            return path
    return current

PROJECT_ROOT = find_project_root()
load_dotenv(PROJECT_ROOT / '.env', override=False)

SAMPLE_PDF = PROJECT_ROOT / 'raw' / '北京市密云水库防御洪水方案.pdf'
if not SAMPLE_PDF.exists():
    raise FileNotFoundError(SAMPLE_PDF)

doc_id = sha1(SAMPLE_PDF.read_bytes()).hexdigest()[:16]
generated_dir = PROJECT_ROOT / 'notebooks' / 'rag' / 'generated' / doc_id
chunks_json_path = generated_dir / 'chunks.json'
triples_json_path = generated_dir / 'triples.json'

if not chunks_json_path.exists():
    raise FileNotFoundError(f'请先执行第三课生成 chunks.json: {chunks_json_path}')

chunks = json.loads(chunks_json_path.read_text(encoding='utf-8'))

print('doc_id:', doc_id)
print('chunk_count:', len(chunks))
print('triples_json_path:', triples_json_path)

doc_id: 63b7d4d0675426b5
chunk_count: 143
triples_json_path: /home/dev/bxc/fastapi-study/notebooks/rag/generated/63b7d4d0675426b5/triples.json


## 4. 配置模型客户端

本课使用大模型网关：

```text
base_url: http://192.168.102.19:8082/v1
model: qwen2.5-0.5b-instruct
```

它是 OpenAI-compatible 接口，所以可以用 OpenAI SDK 调用。

In [29]:
RAG_CONFIG = {
    'model_base_url': os.getenv('RAG_MODEL_BASE_URL', 'http://192.168.102.19:8082/v1'),
    'chat_model': os.getenv('RAG_CHAT_MODEL', 'qwen2.5-0.5b-instruct'),
}

client = OpenAI(api_key=os.getenv('RAG_MODEL_API_KEY', 'EMPTY'), base_url=RAG_CONFIG['model_base_url'])
pprint(RAG_CONFIG)

{'chat_model': 'qwen2.5-0.5b-instruct',
 'model_base_url': 'http://192.168.102.19:8082/v1'}


## 5. 选择适合抽取的 chunk

不是所有 chunk 都适合抽取三元组。

例如目录页、页码、空白页、很短的标题，都可能产生低质量关系。

第一版先过滤太短的 chunk：

```text
char_count >= 80
```

In [30]:
candidate_chunks = [chunk for chunk in chunks if chunk.get('char_count', 0) >= 80]

print('all chunks:', len(chunks))
print('candidate chunks:', len(candidate_chunks))

for chunk in candidate_chunks[30:38]:
    print('=' * 80)
    print(chunk['chunk_id'], 'page=', chunk['page_start'], 'chars=', chunk['char_count'])
    print(chunk['text'][:240].replace('\n', ' '))

all chunks: 143
candidate chunks: 122
63b7d4d0675426b5_chunk_0035 page= 30 chars= 280
成。  6.2.3 会商程序与内容  当密云水库水位接近汛限水位或气象部门发布极端强  降雨天气预警信息时，由调度运行科报处水旱灾害防御工作  主管领导，经同意后启动水旱灾害防御会商。  会商时，调度运行科汇报当前雨水情，工程管理科汇报  工程运行情况，之后由处技术负责人与调度运行科、工程管  理科等相关单位商讨制定调度方案，水旱灾害防御工作领导  小组讨论是否同意方案。讨论通过后，由密云水库管理处主  任签发后，报北京市水务局批准，报北京市水利工程管理中  心备案。  图 
63b7d4d0675426b5_chunk_0036 page= 31 chars= 615
密云水库管理处水旱灾害防御工作领导小组组织召开  水旱灾害防御会商，根据当前雨水工情信息，制定洪水调度  措施。  6.3.2 请示、审批程序  由密云水库管理处调度运行科起草水旱灾害防御调度  请示，由密云水库管理处主任签发后，上报北京市水务局批  准，同时抄送市水利工程管理中心，批准后将洪水调度措施  通报水库下游各区防汛指挥部办公室。密云区政府防汛抗旱  （应急）指挥部办公室（联系人：张立涛；电话18500937277、 69044808；传真：69040309）；潮白
63b7d4d0675426b5_chunk_0037 page= 32 chars= 506
科技推广中心保障雨、水、工情遥测系统正常运行；确  保报汛通讯通畅和报汛网络通畅。  后勤服务中心负责交通保障，确保水旱灾害防御用车及  抢险物资运输。  密云水库管理处水旱灾害防御工作领导小组加强与密  云区水务局对接联动，及时水库下游属地政府通报水旱灾害  防御工作信息，协助做好水旱灾害抢险救灾技术支撑工作，  及时报告防洪调度方案和水库水雨情信息，泄洪期间加强信  息反馈及共享。  水旱灾害防御调度请示，见附表1； 水旱灾害防御调度通知单，见附表2； 密云水库防汛调度令
63b7d4d0675426b5_chunk_0038 page= 33 chars= 740
流潮河、白河汇流而成，控制潮白河流域面积15788km2，占 总流域面积的88％。潮河发源于河

## 6. 设计抽取 prompt

对小模型，prompt 必须明确：

```text
只输出 JSON
固定字段
不要 Markdown
不要编造
证据必须来自原文
```

但即使这样，小模型仍然可能漏字段。所以后面还要做代码校验。

In [31]:
SYSTEM_PROMPT = (
    '你是一个中文知识图谱三元组抽取器。\n'
    '你必须只输出合法 JSON，不要输出 Markdown，不要解释。\n\n'
    '输出格式固定为：\n'
    '{\n'
    '  "triples": [\n'
    '    {\n'
    '      "subject": "",\n'
    '      "predicate": "",\n'
    '      "object": "",\n'
    '      "evidence": "",\n'
    '      "confidence": 0.0\n'
    '    }\n'
    '  ]\n'
    '}\n\n'
    '规则：\n'
    '1. subject、predicate、object 必须来自原文，或由原文中的指代直接消解得到。\n'
    '2. evidence 必须是原文中的短句。\n'
    '3. 不要编造原文没有的信息。\n'
    '4. 没有三元组时输出 {"triples": []}。\n'
    '5. 每个 triple 必须包含 subject、predicate、object、evidence、confidence 五个字段。'
)

FEW_SHOT_MESSAGES = [
    {
        'role': 'user',
        'content': '示例文本：密云水库管理处发布泄洪通知，通知要求下游乡镇做好防汛准备。',
    },
    {
        'role': 'assistant',
        'content': json.dumps(
            {
                'triples': [
                    {
                        'subject': '密云水库管理处',
                        'predicate': '发布',
                        'object': '泄洪通知',
                        'evidence': '密云水库管理处发布泄洪通知',
                        'confidence': 0.95,
                    },
                    {
                        'subject': '泄洪通知',
                        'predicate': '要求',
                        'object': '下游乡镇做好防汛准备',
                        'evidence': '通知要求下游乡镇做好防汛准备',
                        'confidence': 0.90,
                    },
                ]
            },
            ensure_ascii=False,
        ),
    },
]

print(SYSTEM_PROMPT[:300])

你是一个中文知识图谱三元组抽取器。
你必须只输出合法 JSON，不要输出 Markdown，不要解释。

输出格式固定为：
{
  "triples": [
    {
      "subject": "",
      "predicate": "",
      "object": "",
      "evidence": "",
      "confidence": 0.0
    }
  ]
}

规则：
1. subject、predicate、object 必须来自原文，或由原文中的指代直接消解得到。
2. evidence 必须是原文中的短句。
3. 不要编造原文没有的信息


## 7. 调用模型抽取一个 chunk

先抽一个 chunk，观察模型原始输出。

不要急着批量处理。先确认输出格式能被解析。

In [32]:
def call_triple_extractor(text: str, max_chars: int = 900) -> str:
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        *FEW_SHOT_MESSAGES,
        {'role': 'user', 'content': '请抽取下面文本中的三元组：\n' + text[:max_chars]},
    ]
    response = client.chat.completions.create(
        model=RAG_CONFIG['chat_model'],
        temperature=0,
        messages=messages,
    )
    return response.choices[0].message.content.strip()

sample_chunk = candidate_chunks[0]
raw_output = call_triple_extractor(sample_chunk['text'])

print('chunk_id:', sample_chunk['chunk_id'])
print('raw model output:')
print(raw_output)

chunk_id: 63b7d4d0675426b5_chunk_0002
raw model output:
```json
{
  "triples": [
    {
      "subject": "目录",
      "predicate": "内容",
      "object": "目 录",
      "evidence": "目录",
      "confidence": 1.0
    },
    {
      "subject": "2024 年北京市密云水库洪水调度方案",
      "predicate": "内容",
      "object": "2024 年北京市密云水库洪水调度方案",
      "evidence": "2024 年北京市密云水库洪水调度方案",
      "confidence": 0.98
    },
    {
      "subject": "2024 年北京市密云水库防洪抢险预案",
      "predicate": "内容",
      "object": "2024 年北京市密云水库防洪抢险预案",
      "evidence": "2024 年北京市密云水库防洪抢险预案",
      "confidence": 0.97
    }
  ]
}
```


## 8. 解析模型 JSON

小模型有时会输出：

```text
Markdown 代码块
JSON 前后多余文字
字段缺失
confidence 缺失
```

所以先写一个宽容的 JSON 提取函数。

In [33]:
def extract_json_object(raw: str) -> dict:
    text = raw.strip()
    if text.startswith('```'):
        text = re.sub(r'^```(?:json)?', '', text).strip()
        text = re.sub(r'```$', '', text).strip()

    try:
        return json.loads(text)
    except json.JSONDecodeError:
        match = re.search(r'\{.*\}', text, flags=re.DOTALL)
        if not match:
            raise
        return json.loads(match.group(0))

parsed_output = extract_json_object(raw_output)
pprint(parsed_output)

{'triples': [{'confidence': 1.0,
              'evidence': '目录',
              'object': '目 录',
              'predicate': '内容',
              'subject': '目录'},
             {'confidence': 0.98,
              'evidence': '2024 年北京市密云水库洪水调度方案',
              'object': '2024 年北京市密云水库洪水调度方案',
              'predicate': '内容',
              'subject': '2024 年北京市密云水库洪水调度方案'},
             {'confidence': 0.97,
              'evidence': '2024 年北京市密云水库防洪抢险预案',
              'object': '2024 年北京市密云水库防洪抢险预案',
              'predicate': '内容',
              'subject': '2024 年北京市密云水库防洪抢险预案'}]}


## 9. 校验和补齐三元组

这是本课最重要的工程控制点。

不要直接把模型输出写进 Neo4j。

必须做：

```text
字段存在性检查
subject/predicate/object 非空
confidence 转数字
evidence 缺失时做保守补齐
追加 doc_id/chunk_id/page_start/page_end
```

In [34]:
def normalize_confidence(value, default: float = 0.5) -> float:
    try:
        score = float(value)
    except (TypeError, ValueError):
        score = default
    return max(0.0, min(1.0, score))


def validate_and_enrich_triples(raw_data: dict, chunk: dict) -> list[dict]:
    triples = raw_data.get('triples', [])
    if not isinstance(triples, list):
        return []

    valid = []
    chunk_text = chunk.get('text', '')
    for item in triples:
        if not isinstance(item, dict):
            continue
        subject = str(item.get('subject', '')).strip()
        predicate = str(item.get('predicate', '')).strip()
        obj = str(item.get('object', '')).strip()
        evidence = str(item.get('evidence', '')).strip()
        confidence = normalize_confidence(item.get('confidence', 0.5))

        if not subject or not predicate or not obj:
            continue
        if len(predicate) > 30:
            continue

        if not evidence:
            evidence = chunk_text[:120].replace('\n', ' ')

        valid.append(
            {
                'subject': subject,
                'predicate': predicate,
                'object': obj,
                'evidence': evidence,
                'confidence': confidence,
                'doc_id': chunk['doc_id'],
                'chunk_id': chunk['chunk_id'],
                'page_start': chunk['page_start'],
                'page_end': chunk['page_end'],
                'file_name': chunk['file_name'],
            }
        )
    return valid

sample_triples = validate_and_enrich_triples(parsed_output, sample_chunk)
pprint(sample_triples)

[{'chunk_id': '63b7d4d0675426b5_chunk_0002',
  'confidence': 1.0,
  'doc_id': '63b7d4d0675426b5',
  'evidence': '目录',
  'file_name': '北京市密云水库防御洪水方案.pdf',
  'object': '目 录',
  'page_end': 3,
  'page_start': 3,
  'predicate': '内容',
  'subject': '目录'},
 {'chunk_id': '63b7d4d0675426b5_chunk_0002',
  'confidence': 0.98,
  'doc_id': '63b7d4d0675426b5',
  'evidence': '2024 年北京市密云水库洪水调度方案',
  'file_name': '北京市密云水库防御洪水方案.pdf',
  'object': '2024 年北京市密云水库洪水调度方案',
  'page_end': 3,
  'page_start': 3,
  'predicate': '内容',
  'subject': '2024 年北京市密云水库洪水调度方案'},
 {'chunk_id': '63b7d4d0675426b5_chunk_0002',
  'confidence': 0.97,
  'doc_id': '63b7d4d0675426b5',
  'evidence': '2024 年北京市密云水库防洪抢险预案',
  'file_name': '北京市密云水库防御洪水方案.pdf',
  'object': '2024 年北京市密云水库防洪抢险预案',
  'page_end': 3,
  'page_start': 3,
  'predicate': '内容',
  'subject': '2024 年北京市密云水库防洪抢险预案'}]


## 10. 批量抽取少量 chunk

为了避免一次性调用太多模型请求，默认只处理前 3 个候选 chunk。

如果后续想全量抽取，再显式修改：

```python
MAX_CHUNKS_TO_EXTRACT = len(candidate_chunks)
```

In [19]:
MAX_CHUNKS_TO_EXTRACT = 28

all_triples = []
failed_chunks = []

for index, chunk in enumerate(candidate_chunks[20:MAX_CHUNKS_TO_EXTRACT], start=1):
    print(f'extracting {index}/{MAX_CHUNKS_TO_EXTRACT}:', chunk['chunk_id'])
    try:
        raw = call_triple_extractor(chunk['text'])
        data = extract_json_object(raw)
        triples = validate_and_enrich_triples(data, chunk)
        all_triples.extend(triples)
        print('  triples:', len(triples))
    except Exception as exc:
        failed_chunks.append({'chunk_id': chunk['chunk_id'], 'error': f'{type(exc).__name__}: {exc}'})
        print('  failed:', type(exc).__name__, exc)

print('total triples:', len(all_triples))
print('failed chunks:', len(failed_chunks))

extracting 1/28: 63b7d4d0675426b5_chunk_0024
  triples: 19
extracting 2/28: 63b7d4d0675426b5_chunk_0025
  failed: JSONDecodeError Expecting ',' delimiter: line 1 column 17533 (char 17532)
extracting 3/28: 63b7d4d0675426b5_chunk_0027
  failed: JSONDecodeError Expecting ',' delimiter: line 1 column 2353 (char 2352)
extracting 4/28: 63b7d4d0675426b5_chunk_0028
  failed: JSONDecodeError Expecting ',' delimiter: line 1 column 123 (char 122)
extracting 5/28: 63b7d4d0675426b5_chunk_0029
  failed: JSONDecodeError Expecting ',' delimiter: line 1 column 2217 (char 2216)
extracting 6/28: 63b7d4d0675426b5_chunk_0030
  triples: 4
extracting 7/28: 63b7d4d0675426b5_chunk_0031
  triples: 0
extracting 8/28: 63b7d4d0675426b5_chunk_0032
  failed: AttributeError 'list' object has no attribute 'get'
total triples: 23
failed chunks: 5


## 11. 观察抽取结果

重点看三个问题：

```text
关系是否来自原文
predicate 是否太泛化
evidence 是否可追溯
confidence 是否合理
```

In [35]:
for triple in all_triples[20:28]:
    print('=' * 80)
    print(f"{triple['subject']} --{triple['predicate']}--> {triple['object']}")
    print('confidence:', triple['confidence'])
    print('chunk_id:', triple['chunk_id'], 'page:', triple['page_start'])
    print('evidence:', triple['evidence'][:180])

if failed_chunks:
    print('failed chunks:')
    pprint(failed_chunks)

7孔桥节制闸调度 --调度--> 密云水
confidence: 0.95
chunk_id: 63b7d4d0675426b5_chunk_0030 page: 25
evidence: 密云水库管理处发布泄洪通知
调洪 --调度--> 调节池
confidence: 0.9
chunk_id: 63b7d4d0675426b5_chunk_0030 page: 25
evidence: 调节池挡水闸闸门全开,调节池最高水位不超过90.50m
调节池 --调度--> 京密引水渠
confidence: 0.9
chunk_id: 63b7d4d0675426b5_chunk_0030 page: 25
evidence: 京密引水渠引水流量不能满足泄洪要求
failed chunks:
[{'chunk_id': '63b7d4d0675426b5_chunk_0025',
  'error': "JSONDecodeError: Expecting ',' delimiter: line 1 column 17533 "
           '(char 17532)'},
 {'chunk_id': '63b7d4d0675426b5_chunk_0027',
  'error': "JSONDecodeError: Expecting ',' delimiter: line 1 column 2353 (char "
           '2352)'},
 {'chunk_id': '63b7d4d0675426b5_chunk_0028',
  'error': "JSONDecodeError: Expecting ',' delimiter: line 1 column 123 (char "
           '122)'},
 {'chunk_id': '63b7d4d0675426b5_chunk_0029',
  'error': "JSONDecodeError: Expecting ',' delimiter: line 1 column 2217 (char "
           '2216)'},
 {'chunk_id': '63b7d4d0675426b5_chunk_0032',
  'error': "Attrib

## 12. 保存 triples.json

保存路径：

```text
notebooks/rag/generated/{doc_id}/triples.json
```

下一课会读取这个文件，写入 Neo4j。

In [36]:
triple_payload = {
    'doc_id': doc_id,
    'source_chunks': [chunk['chunk_id'] for chunk in candidate_chunks[:MAX_CHUNKS_TO_EXTRACT]],
    'triple_count': len(all_triples),
    'failed_chunks': failed_chunks,
    'triples': all_triples,
}

triples_json_path.write_text(json.dumps(triple_payload, ensure_ascii=False, indent=2), encoding='utf-8')

print('triples_json_path:', triples_json_path)
print('size KB:', round(triples_json_path.stat().st_size / 1024, 2))

triples_json_path: /home/dev/bxc/fastapi-study/notebooks/rag/generated/63b7d4d0675426b5/triples.json
size KB: 12.1


## 13. 本课小结

本课完成：

```text
chunks.json
-> qwen2.5-0.5b-instruct
-> raw JSON
-> parse JSON
-> validate and enrich triples
-> triples.json
```

关键结论：

```text
三元组抽取不是只调用模型。
真正可靠的流程是：模型抽取 + 结构校验 + 证据约束 + 失败记录。
```

下一课会进入：

```text
triples.json
-> Neo4j Document / Chunk / Entity / RELATION
```

## 14. 练习

请你观察本课输出后回答：

1. 哪些三元组看起来是高质量的？为什么？
2. 有没有 predicate 太泛化的情况，比如“是”“包括”“做好”？
3. 为什么 `evidence` 不能省略？
4. 为什么第一版默认只抽取 3 个 chunk，而不是全量 143 个 chunk？